<center>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</center>

----------

<h1 align="center"> Exercise - Pandas Data Processing </h1>


----------

# Exercise 1
Load all the data files into Python

In [ ]:
import pandas as pd
import csv

In [ ]:
sales = pd.read_csv('https://weclouddata.s3.amazonaws.com/datasets/retail/adventureworks/csv/DimSalesOrder.csv')
product = pd.read_csv('https://weclouddata.s3.amazonaws.com/datasets/retail/adventureworks/csv/DimProduct.csv')
category = pd.read_csv('https://weclouddata.s3.amazonaws.com/datasets/retail/adventureworks/csv/DimProductCategory.csv')
subcategory = pd.read_csv('https://weclouddata.s3.amazonaws.com/datasets/retail/adventureworks/csv/DimProductSubCategory.csv')
customer = pd.read_csv('https://weclouddata.s3.amazonaws.com/datasets/retail/adventureworks/csv/DimCustomer.csv')
reseller = pd.read_csv('https://weclouddata.s3.amazonaws.com/datasets/retail/adventureworks/csv/DimReseller.csv')
date = pd.read_csv('https://weclouddata.s3.amazonaws.com/datasets/retail/adventureworks/csv/DimDate.csv')
geography = pd.read_csv('https://weclouddata.s3.amazonaws.com/datasets/retail/adventureworks/csv/DimGeography.csv')
territory = pd.read_csv('https://weclouddata.s3.amazonaws.com/datasets/retail/adventureworks/csv/DimSalesTerritory.csv')

Look at each table to make sure they have been read the way you wanted

In [ ]:
reseller

In [ ]:
# The reseller file is semicolon separated, but reading it with sep=';'
# directly leaves row 700 misaligned. Requires some tinkering:
reseller = pd.read_csv('https://weclouddata.s3.amazonaws.com/datasets/retail/adventureworks/csv/DimReseller.csv', sep=';')
reseller

In [ ]:
# Fixed: read each line as a single column, then split on ';'

reseller = pd.read_csv('https://weclouddata.s3.amazonaws.com/datasets/retail/adventureworks/csv/DimReseller.csv', sep=',', header=None)
reseller =  reseller.loc[:,0].str.split(';', expand=True)
reseller

In [ ]:
new_header = reseller.iloc[0] #grab the first row for the header
reseller = reseller[1:] #take the data less the header row
reseller.columns = new_header #set the header row as the df header
reseller

# Exercise 2
Which table should be used as the fact table?

fact table: contains all the data to be analyzed

dimension table: contains all the data which the data in the fact table can be analyzed by

In [ ]:
sales.head()

Check if there are any empty cells in this table

In [ ]:
sales.isna()

Checking the NA values:

In [ ]:
# isna() - for checking the NA values
sales.isna().sum()

Remove columns you don't think are useful

In [ ]:
sales = sales.drop(['TaxAmount','FreightAmount','CarrierTrackingNumber','RevisionNumber','EmployeeKey','SalesOrderLineKey','SalesOrderLineNumber','CurrencyKey','CustomerPONumber','DiscountAmount','ProductStandardCost','PromotionKey','DueDateKey','ShipDateKey','ExtendedAmount','UnitPriceDiscountPct'],axis = 1)
sales

# Exercise 3
Calculate total sales, costs, and gross profit

In [ ]:
# Total sales
total_sales = sales['SalesAmount'].sum()
total_sales

In [ ]:
# Total costs
total_costs = sales['TotalProductCost'].sum()
total_costs

In [ ]:
# Total gross profit = total sales - total costs
total_gprofit = total_sales - total_costs
total_gprofit

Make a new column in sales which contains gross profit by row

In [ ]:
sales['GrossProfit'] = sales['SalesAmount'] - sales['TotalProductCost']

In [ ]:
sales

# Exercise 4
Find the top 5 largest orders placed by direct customers

In [ ]:
sales['CustomerKey']

In [ ]:
# Filter out resellers
# (rows where CustomerKey is -1 are reseller orders, not direct customers)
sales_customer = sales[sales['CustomerKey'] != -1]
sales_customer

In [ ]:
# How often does each direct customer appear? The length of this result is
# also the number of distinct direct customers
sales_customer['CustomerKey'].value_counts()

In [ ]:
# METHOD 1: Using sort_values()
sales_customer.sort_values(by='SalesAmount', ascending=False)[
    ['CustomerKey', 'SalesAmount', 'GrossProfit']].head(5)

In [ ]:
# METHOD 2: Using rank()
sales_customer['AmountRank'] = sales_customer['SalesAmount'].rank(ascending=False)

sales_customer[sales_customer['AmountRank'] <= 5][
    ['CustomerKey', 'SalesAmount', 'GrossProfit', 'AmountRank']]

Additional Exercise: Find the first and last names of the customers who placed these orders

In [ ]:
# Looking up a single customer with a boolean mask
customer[customer['CustomerKey'] == 11433][['CustomerKey', 'FirstName', 'LastName']]

In [ ]:
# Looking up all five at once: isin() builds a boolean mask that is True
# wherever CustomerKey matches any value in the list
top_keys = sales_customer.sort_values(by='SalesAmount', ascending=False)['CustomerKey'].head(5)

customer[customer['CustomerKey'].isin(top_keys)][['CustomerKey', 'FirstName', 'LastName']]